# 01-1. Gene Mutation Encoding

각 유전자 변이 상태를 기능적 의미에 따라 3단계로 인코딩하고 Parquet으로 저장한다.

| 값 | 의미 |
|---|---|
| 0 | Wild Type (WT) |
| 1 | Synonymous Mutation (동의 변이 — 단백질 변화 없음) |
| 2 | Functional Mutation (Missense / Nonsense / Frameshift 등) |

In [11]:
import sys
from pathlib import Path

ROOT = Path("__file__").resolve().parents[1]
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

DATA_RAW  = ROOT / "data" / "raw"
DATA_OUT  = ROOT / "data" / "process"
DATA_OUT.mkdir(parents=True, exist_ok=True)

In [12]:
import pandas as pd
from cancer_hack.features_basic import encode_mutation, make_mutation_encoding
from cancer_hack.io import save_parquet, load_parquet

## 1. 인코딩 함수 단위 검증

In [13]:
test_cases = {
    # 단일 변이
    "WT":           0,  # Wild Type
    "R895R":        1,  # Synonymous
    "L42L":         1,  # Synonymous
    "R132H":        2,  # Missense
    "V600E":        2,  # Missense
    "E237*":        2,  # Nonsense
    "K16fs":        2,  # Frameshift
    "L1854fs":      2,  # Frameshift
    # 복수 변이 (공백 구분) — 최대 심각도(max) 반환
    "F157L S1042F": 2,  # Functional + Functional → 2
    "S622S G827R":  2,  # Synonymous + Functional → 2
    "G179G A408A":  1,  # Synonymous + Synonymous → 1
    "WT WT":        0,  # WT + WT → 0
    "Q924* P44P":   2,  # Nonsense + Synonymous → 2
    "H22H G49W":    2,  # Synonymous + Missense → 2
    "I758S P700fs": 2,  # Missense + Frameshift → 2
}

all_pass = True
for val, expected in test_cases.items():
    result = encode_mutation(val)
    status = "PASS" if result == expected else "FAIL"
    if status == "FAIL":
        all_pass = False
    print(f"{status}  encode_mutation({val!r:<30}) = {result}  (expected {expected})")

assert all_pass, "인코딩 함수 검증 실패"
print("\n모든 케이스 통과")

PASS  encode_mutation('WT'                          ) = 0  (expected 0)
PASS  encode_mutation('R895R'                       ) = 1  (expected 1)
PASS  encode_mutation('L42L'                        ) = 1  (expected 1)
PASS  encode_mutation('R132H'                       ) = 2  (expected 2)
PASS  encode_mutation('V600E'                       ) = 2  (expected 2)
PASS  encode_mutation('E237*'                       ) = 2  (expected 2)
PASS  encode_mutation('K16fs'                       ) = 2  (expected 2)
PASS  encode_mutation('L1854fs'                     ) = 2  (expected 2)
PASS  encode_mutation('F157L S1042F'                ) = 2  (expected 2)
PASS  encode_mutation('S622S G827R'                 ) = 2  (expected 2)
PASS  encode_mutation('G179G A408A'                 ) = 1  (expected 1)
PASS  encode_mutation('WT WT'                       ) = 0  (expected 0)
PASS  encode_mutation('Q924* P44P'                  ) = 2  (expected 2)
PASS  encode_mutation('H22H G49W'                   ) = 2  (expe

## 2. 원본 데이터 로드

In [14]:
train_raw = pd.read_csv(DATA_RAW / "train.csv")
print(f"train shape: {train_raw.shape}")
print(f"SUBCLASS 분포:\n{train_raw['SUBCLASS'].value_counts()}")

train shape: (6201, 4386)
SUBCLASS 분포:
SUBCLASS
BRCA      786
KIPAN     515
GBMLGG    461
STES      379
KIRC      334
THCA      324
SKCM      276
PRAD      266
OV        253
LGG       229
HNSC      223
COAD      223
SARC      198
UCEC      198
LUAD      184
LUSC      178
LIHC      158
LAML      158
CESC      155
PCPG      147
TGCT      124
PAAD      120
BLCA      104
THYM       98
ACC        72
DLBC       38
Name: count, dtype: int64


## 2-1. 복수 변이 셀 현황 파악

하나의 셀에 공백으로 구분된 여러 변이가 입력된 경우가 존재한다.  
인코딩 시 각 토큰을 개별 인코딩한 뒤 **최대 심각도(max)** 를 취한다.
- Synonymous + Functional → **2** (Functional 우선)
- Synonymous + Synonymous → **1**
- WT + WT → **0**

In [15]:
gene_cols_raw = [c for c in train_raw.columns if c not in {"ID", "SUBCLASS"}]
all_vals = train_raw[gene_cols_raw].stack()

multi_vals = all_vals[all_vals.str.contains(" ", na=False)]
token_counts = multi_vals.str.split().str.len()

print(f"공백 포함 셀 수  : {len(multi_vals):,}  ({len(multi_vals) / len(all_vals) * 100:.2f}%)")
print(f"전체 셀 수       : {len(all_vals):,}")
print()
print("토큰 수 분포 (상위 10):")
print(token_counts.value_counts().sort_index().head(10).to_string())
print()
print("샘플 복수 변이 값 (5개):")
for v in multi_vals.unique()[:5]:
    tokens = v.split()
    encodings = [encode_mutation(t) for t in tokens]
    print(f"  {v!r:<35}  토큰별={encodings}  → max={max(encodings)}")

공백 포함 셀 수  : 22,026  (0.08%)
전체 셀 수       : 27,185,184

토큰 수 분포 (상위 10):
2     16261
3      3023
4      1399
5       382
6       374
7        79
8       159
9        25
10       82
11       16

샘플 복수 변이 값 (5개):
  'Q369* I368N'                        토큰별=[2, 2]  → max=2
  'S622S G827R'                        토큰별=[1, 2]  → max=2
  'E412K R1800C'                       토큰별=[2, 2]  → max=2
  'D3546V G3739G'                      토큰별=[2, 1]  → max=2
  'A368V E646K'                        토큰별=[2, 2]  → max=2


## 3. 3단계 인코딩 적용

In [16]:
train_enc = make_mutation_encoding(train_raw)
print(f"인코딩 후 shape: {train_enc.shape}")
print(f"dtypes 샘플:\n{train_enc.dtypes.value_counts()}")

인코딩 후 shape: (6201, 4386)
dtypes 샘플:
int8    4384
str        2
Name: count, dtype: int64


## 4. 인코딩 결과 검증

In [17]:
gene_cols = [c for c in train_enc.columns if c not in {"ID", "SUBCLASS"}]

# 값이 0 / 1 / 2 외에 없어야 한다
unique_vals = set(train_enc[gene_cols].stack().unique())
assert unique_vals <= {0, 1, 2}, f"예상치 못한 값: {unique_vals - {0, 1, 2}}"

# 분포 확인
val_counts = train_enc[gene_cols].stack().value_counts().sort_index()
val_pct    = (val_counts / val_counts.sum() * 100).round(2)

print("값 분포:")
for v, cnt, pct in zip(val_counts.index, val_counts, val_pct):
    label = {0: "WT", 1: "Synonymous", 2: "Functional"}[v]
    print(f"  {v} ({label:>10s}): {cnt:>10,}  ({pct:.2f}%)")

print("\n인코딩 검증 완료")

값 분포:
  0 (        WT): 26,966,291  (99.19%)
  1 (Synonymous):     52,896  (0.19%)
  2 (Functional):    165,997  (0.61%)

인코딩 검증 완료


In [18]:
# ID, SUBCLASS 열 보존 확인
assert "ID" in train_enc.columns
assert "SUBCLASS" in train_enc.columns
assert train_enc["ID"].equals(train_raw["ID"])
assert train_enc["SUBCLASS"].equals(train_raw["SUBCLASS"])
print("ID / SUBCLASS 열 보존 확인 완료")

ID / SUBCLASS 열 보존 확인 완료


## 5. Parquet 저장

In [19]:
out_path = DATA_OUT / "train_mutation_encoded.parquet"
save_parquet(train_enc, out_path)

# 저장 확인: 다시 읽어서 shape 비교
check = load_parquet(out_path)
assert check.shape == train_enc.shape, "저장된 파일과 shape 불일치"

print(f"저장 완료: {out_path}")
print(f"  파일 크기: {out_path.stat().st_size / 1024 / 1024:.1f} MB")
print(f"  shape    : {check.shape}")

저장 완료: /Users/hangeumjun/onco-ai/data/process/train_mutation_encoded.parquet
  파일 크기: 3.0 MB
  shape    : (6201, 4386)


In [20]:
df = load_parquet(DATA_OUT / "train_mutation_encoded.parquet")

df.head(20)

,ID,SUBCLASS,A2M,AAAS,AADAT,AARS1,ABAT,ABCA1,ABCA2,ABCA3,...,ZNF292,ZNF365,ZNF639,ZNF707,ZNFX1,ZNRF4,ZPBP,ZW10,ZWINT,ZYX
0,TRAIN_0000,KIPAN,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,TRAIN_0001,SARC,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,TRAIN_0002,SKCM,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,TRAIN_0003,KIRC,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,TRAIN_0004,GBMLGG,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,TRAIN_0005,STES,0,0,2,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
6,TRAIN_0006,BRCA,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,TRAIN_0007,THCA,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,TRAIN_0008,LIHC,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,TRAIN_0009,STES,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
